# Quantum Computing Basics

This notebook is a self-contained reference for the notation, symbols, and
operations used throughout all other notebooks in this repository.
Each concept is illustrated with a concrete numeric example computed with
SymPy's quantum-physics module.

---
## 1. The qubit

A **qubit** is the basic unit of quantum information.  In the **computational
basis** it can be in one of two orthonormal basis states:

$$|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}, \qquad
  |1\rangle = \begin{pmatrix}0\\1\end{pmatrix}$$

A general single-qubit **pure state** is a normalised complex linear combination:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad |\alpha|^2 + |\beta|^2 = 1$$

The **bra** $\langle\psi|$ is the conjugate-transpose of the ket $|\psi\rangle$.

Two frequently used single-qubit states in the **Hadamard basis** are:

$$|{+}\rangle = \frac{|0\rangle+|1\rangle}{\sqrt{2}}, \qquad
  |{-}\rangle = \frac{|0\rangle-|1\rangle}{\sqrt{2}}$$

In [1]:
"""Introductory notebook: quantum computing notation and basic operations."""

from sympy import sqrt, Rational, simplify, Matrix
from sympy.physics.quantum.qubit import Qubit, measure_all
from sympy.physics.quantum.gate import HadamardGate, XGate, ZGate, CNotGate, PhaseGate
from sympy.physics.quantum.qapply import qapply
from sympy.physics.quantum import TensorProduct, Dagger, InnerProduct
from sympy.physics.quantum.represent import represent
from sympy.physics.quantum.qubit import matrix_to_qubit

# |0> and |1> in column-vector form
ket0 = Qubit("0")
ket1 = Qubit("1")
print("|0> =", represent(ket0).T, "  (column vector)")
print("|1> =", represent(ket1).T)

|0> = Matrix([[1, 0]])   (column vector)
|1> = Matrix([[0, 1]])


---
## 2. Multi-qubit states and tensor products

The **tensor product** $\otimes$ combines two quantum systems into a larger one.

- $|a\rangle \otimes |b\rangle$ -- explicit tensor product of $|a\rangle$ and $|b\rangle$
- $|ab\rangle$ -- shorthand for $|a\rangle \otimes |b\rangle$
- $|0\rangle^{\otimes n}$ or $|0\rangle^n$ -- $n$-fold tensor product $|0\rangle \otimes \cdots \otimes |0\rangle$
- $|0\rangle^n|0\rangle^n$ -- two $n$-qubit registers both initialised to zero

For $n=2$ the computational basis has $2^2 = 4$ elements:

$$|00\rangle,\quad |01\rangle,\quad |10\rangle,\quad |11\rangle$$

and the full state vector lives in $\mathbb{C}^{2^n}$.

**Numeric example** -- $|0\rangle \otimes |1\rangle = |01\rangle$:

In [2]:
# |0> tensor |1>
ket01_tp = TensorProduct(ket0, ket1)
ket01 = Qubit("01")

print("|0> x |1> (TensorProduct) =", represent(ket01_tp).T)
print("|01>                      =", represent(ket01).T)
print("Equal:", represent(ket01_tp) == represent(ket01))

|0> x |1> (TensorProduct) = Matrix([[0, 1, 0, 0]])
|01>                      = Matrix([[0, 1, 0, 0]])
Equal: True


---
## 3. Fundamental single-qubit gates

Quantum gates are **unitary matrices** $U$ ($U^\dagger U = I$).

- **Pauli-X (NOT)** $X = \begin{pmatrix}0&1\\1&0\end{pmatrix}$ -- $X|0\rangle=|1\rangle$, $X|1\rangle=|0\rangle$
- **Pauli-Z** $Z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}$ -- $Z|0\rangle=|0\rangle$, $Z|1\rangle=-|1\rangle$
- **Hadamard** $H = \frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ -- $H|0\rangle=|{+}\rangle$, $H|1\rangle=|{-}\rangle$

**Numeric example** -- applying $H$ to $|0\rangle$:

In [3]:
# H|0> = |+> = (|0> + |1>) / sqrt(2)
H0 = qapply(HadamardGate(0) * ket0)
print("H|0> =", H0)
print("      (column vector):", represent(H0).T)

# H|1> = |-> = (|0> - |1>) / sqrt(2)
H1 = qapply(HadamardGate(0) * ket1)
print("H|1> =", H1)

H|0> = sqrt(2)*|0>/2 + sqrt(2)*|1>/2
      (column vector): Matrix([[sqrt(2)/2, sqrt(2)/2]])
H|1> = sqrt(2)*|0>/2 - sqrt(2)*|1>/2


---
## 4. The $n$-fold Hadamard $H^{\otimes n}$

Applying $H$ to every qubit of an $n$-qubit $|0\rangle^n$ state creates a **uniform
superposition** over all $2^n$ computational basis states:

$$H^{\otimes n}|0\rangle^n = \frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n-1} |x\rangle$$

More generally, for any $n$-bit string $x$:

$$H^{\otimes n}|x\rangle = \frac{1}{\sqrt{2^n}} \sum_{y=0}^{2^n-1} (-1)^{x \cdot y}|y\rangle$$

where $x \cdot y = x_0 y_0 \oplus x_1 y_1 \oplus \cdots \oplus x_{n-1} y_{n-1}$
is the **bitwise inner product** mod 2.

**Numeric example** -- $H^{\otimes 2}|00\rangle$:

In [4]:
from util.util import hn   # repository helper: n-fold Hadamard operator

# H^2 |00> = (1/2)(|00> + |01> + |10> + |11>)
state = qapply(hn(2) * Qubit("00"))
print("H^2 |00> =", state)
print("Column vector:", represent(state).T)

H^2 |00> = |00>/2 + |01>/2 + |10>/2 + |11>/2
Column vector: Matrix([[1/2, 1/2, 1/2, 1/2]])


---
## 5. The CNOT gate (controlled-NOT)

$$\text{CNOT} = |0\rangle\langle 0| \otimes I + |1\rangle\langle 1| \otimes X
= \begin{pmatrix}1&0&0&0\\0&1&0&0\\0&0&0&1\\0&0&1&0\end{pmatrix}$$

The **control** qubit is unchanged; the **target** qubit is flipped when the
control is $|1\rangle$:

- $|00\rangle \xrightarrow{\text{CNOT}} |00\rangle$
- $|01\rangle \xrightarrow{\text{CNOT}} |01\rangle$
- $|10\rangle \xrightarrow{\text{CNOT}} |11\rangle$
- $|11\rangle \xrightarrow{\text{CNOT}} |10\rangle$

**Numeric example** -- CNOT on all four basis states:

In [5]:
# CNotGate(control_qubit_index, target_qubit_index)
# SymPy qubit ordering: qubit 1 = leftmost (most-significant) bit
cnot = CNotGate(1, 0)   # control = qubit 1, target = qubit 0

for bits in ["00", "01", "10", "11"]:
    out = qapply(cnot * Qubit(bits))
    print(f"CNOT |{bits}> = {out}")

CNOT |00> = |00>
CNOT |01> = |01>
CNOT |10> = |11>
CNOT |11> = |10>


---
## 6. Entanglement: creating a Bell state

Applying $H$ to the first qubit and then CNOT creates the **Bell state**
$|\Phi^+\rangle$, the simplest maximally entangled state:

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

Circuit:
```
|0> --H--*--  }
         |    }  |Phi+>
|0> -----X--  }
```

In [6]:
# Step 1: H on qubit 1 (left qubit of |00>)
after_H = qapply(HadamardGate(1) * Qubit("00"))
print("After H on qubit 1:", after_H)

# Step 2: CNOT with control=1, target=0
bell = qapply(CNotGate(1, 0) * after_H)
print("Bell state |Phi+>  :", bell)
print("Column vector      :", represent(bell).T)

After H on qubit 1: sqrt(2)*|00>/2 + sqrt(2)*|10>/2
Bell state |Phi+>  : sqrt(2)*|00>/2 + sqrt(2)*|11>/2
Column vector      : Matrix([[sqrt(2)/2, 0, 0, sqrt(2)/2]])


---
## 7. Inner product and orthogonality

The **inner product** $\langle\phi|\psi\rangle$ is the overlap between two states.
Orthonormal basis states satisfy:

$$\langle 0|0\rangle = 1, \quad \langle 1|1\rangle = 1, \quad \langle 0|1\rangle = 0$$

The Hadamard-basis states are also orthonormal:

$$\langle{+}|{+}\rangle = 1, \quad \langle{-}|{-}\rangle = 1,
  \quad \langle{+}|{-}\rangle = 0$$

In [7]:
# orthonormality of |0>, |1>
v0 = represent(ket0)
v1 = represent(ket1)

print("<0|0> =", (v0.T * v0)[0, 0])
print("<1|1> =", (v1.T * v1)[0, 0])
print("<0|1> =", (v0.T * v1)[0, 0])

# |+> and |->
vp = represent(qapply(HadamardGate(0) * ket0))   # |+>
vm = represent(qapply(HadamardGate(0) * ket1))   # |->
print("<+|+> =", simplify((vp.T * vp)[0, 0]))
print("<-|-> =", simplify((vm.T * vm)[0, 0]))
print("<+|-> =", simplify((vp.T * vm)[0, 0]))

<0|0> = 1
<1|1> = 1
<0|1> = 0
<+|+> = 1
<-|-> = 1
<+|-> = 0


---
## 8. Measurement

Measuring a qubit in state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$
yields:

- outcome $0$ with probability $|\alpha|^2$, collapsing the state to $|0\rangle$
- outcome $1$ with probability $|\beta|^2$, collapsing the state to $|1\rangle$

**Numeric example** -- measuring $|{+}\rangle = \frac{1}{\sqrt{2}}(|0\rangle+|1\rangle)$
gives each outcome with probability $\frac{1}{2}$:

In [8]:
plus = qapply(HadamardGate(0) * ket0)   # |+>
outcomes = measure_all(plus)
for state, prob in outcomes:
    print(f"  outcome {state}  probability {prob}")

  outcome |0>  probability 1/2
  outcome |1>  probability 1/2


---
## 9. The quantum oracle $U_f$

An **oracle** encodes a classical function $f:\{0,1\}^n \to \{0,1\}^m$
as a unitary transformation acting on an input register $|x\rangle$ and
an ancilla register $|y\rangle$:

$$U_f|x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$$

When $|y\rangle = |0\rangle$ the ancilla simply **stores** $f(x)$:

$$U_f|x\rangle|0\rangle = |x\rangle|f(x)\rangle$$

When $|y\rangle = |{-}\rangle = \frac{|0\rangle-|1\rangle}{\sqrt{2}}$
the ancilla is unchanged but the input register picks up a **phase kick-back**:

$$U_f|x\rangle|{-}\rangle = (-1)^{f(x)}|x\rangle|{-}\rangle$$

This phase kick-back is the mechanism exploited by the Deutsch, Deutsch-Jozsa,
Bernstein-Vazirani, and Simon algorithms.

**Numeric example** -- oracle for $f(x) = x$ (identity) on 1 qubit:

In [9]:
from oracle import oracle

def f_id(x, *args):
    """Identity oracle: f(x) = x (x is a tuple of bit values)."""
    from sympy.physics.quantum.qubit import Qubit as Q, IntQubit
    return IntQubit(Q(*x)).as_int()

for x_str in ["0", "1"]:
    x_in = Qubit(x_str)
    y_in = Qubit("0")
    out = oracle(x_in, y_in, f_id)
    print(f"U_f |{x_str}>|0> = {out}")

U_f |0>|0> = |00>
U_f |1>|0> = |11>


---
## 10. Symbol glossary

- $|\psi\rangle$ -- **ket**: column-vector quantum state
- $\langle\psi|$ -- **bra**: row-vector (conjugate-transpose of the ket)
- $\langle\phi|\psi\rangle$ -- **inner product**: overlap; equals 1 for identical normalised states
- $|\phi\rangle\langle\psi|$ -- **outer product**: rank-1 operator / projector
- $\otimes$ -- **tensor product**: combines two systems into a joint system
- $\oplus$ -- **XOR / addition mod 2**: bitwise exclusive-OR
- $H$ -- **Hadamard gate**: $H|0\rangle=|{+}\rangle$, $H|1\rangle=|{-}\rangle$
- $H^{\otimes n}$ -- **$n$-fold Hadamard**: apply $H$ to every qubit in an $n$-qubit register
- $X$ -- **Pauli-X / NOT**: bit-flip gate
- $Z$ -- **Pauli-Z**: phase-flip gate ($Z|1\rangle = -|1\rangle$)
- CNOT -- **controlled-NOT**: flips target qubit when control is $|1\rangle$
- $U_f$ -- **oracle**: $U_f|x\rangle|y\rangle=|x\rangle|y \oplus f(x)\rangle$
- $U^\dagger$ -- **conjugate-transpose**: inverse of unitary $U$
- $|0\rangle^n$ -- **$n$-qubit zero state**: $|0\rangle^{\otimes n} = |00\cdots0\rangle$
- $\frac{1}{\sqrt{2^n}}\sum_x|x\rangle$ -- **uniform superposition**: equal-amplitude sum over all $2^n$ basis states